In [ ]:
%pip install librosa

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io

import librosa
from IPython.display import Audio

# SVD and Cats


Download the image `cat.jpg`.

In [ ]:
image = io.imread('cat.jpg', as_gray=True)
imgplot = plt.imshow(image, cmap='Greys_r')

Find the singular value decomposition using `np.linalg.svd`. Note that it returns the singular-vector matrices $U$ and $V^{\mathrm T}$ only when `full_matrices=True`.
Output the shapes of `image`, $U$, $\Sigma$, and $V^{\mathrm T}$.

In [ ]:
# your code here
u, s, vh = np.linalg.svd(image, full_matrices=True)
u.shape, s.shape, vh.shape, image.shape
# your code here

Find the approximations $U\Sigma_k V^{\mathrm T}$ for different values of $k$: $1, 2, 5, 10$, etc.
There is no need to rebuild the matrix $\Sigma_k$; it is sufficient to take the first $k$ columns of $U$, truncate $\Sigma$ to $k\times k$, and take the first $k$ rows of $V^{\mathrm T}$. You can use slices; for example, `u[:, :k]`.
Plot the image for each $k$.

In [ ]:
# your code here
for i in (1, 10, 25, 44, 100, 200, 400):
    imgplot = plt.imshow(u[:, 0:i].dot(np.diag(s[0:i])).dot(vh[0:i, :]), cmap='Greys_r')
    plt.show()
# your code here

Find a value of $k$ that is small but preserves sufficient image quality. 

Count the compression ratio of the truncated SVD compared to the initial number of entries in the original image matrix and with the storage cost of the full SVD representation.

In [ ]:
# your code here

[Your conclusions here.]

Now compute a truncated “SVD” without the singular-value matrix (i.e., omit $\Sigma$ in the product) for several values of $k$.

In [ ]:
# your code here
for i in (1, 10, 44, 100, 200, 440):
    imgplot = plt.imshow(u[:, 0:i].dot(vh[0:i, :]), cmap='Greys_r')
    plt.show()
# your code here

What do you see in the images? What is the physical sense of the singular values? Explain.

[Your conclusions here.]

# Bonus task
Implement SVD-based compression of images for coloured pictures.

In [ ]:
# your code here

[Your comments and conclusions here.]

# Fourier Transform and Birds

Download data from the audio file `mixed_audio.mp3`. It is always helpful to look at the data (we will inspect the first second), and in our case also to listen to it (we will do this directly in the notebook!).

In [ ]:
y_mixed, sr_mixed = librosa.load('mixed_audio.mp3', sr=None)

print(f"Audio time series shape: {y_mixed.shape}")
print(f"Sampling rate (sr): {sr_mixed}")
librosa.display.waveshow(y_mixed[:sr_mixed], sr=sr_mixed)
plt.show()

Audio(data=y_mixed, rate=sr_mixed)

What do you hear?

Explain what the sampling rate is.

[Your answers here.]

Now let’s compute the spectrogram. This is where the Fourier transform happens!

In [ ]:
D_mixed = librosa.stft(y_mixed)
S_db_mixed = librosa.amplitude_to_db(np.abs(D_mixed), ref=np.max)

plt.figure(figsize=(12, 6))
librosa.display.specshow(S_db_mixed, sr=sr_mixed, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram')
plt.show()

What is the physical meaning of a spectrogram? The x-axis shows time. In a neighborhood of each point on the x-axis (i.e., over a small time window), the Fourier transform of the audio waveform over that window is computed. The y-axis shows Fourier frequencies; the color indicates the magnitude of the coefficient—the lighter, the larger (see the decibel scale).

What do you see in this spectrogram, and how does it match the audio you heard? Can you locate parts corresponding to different sounds?

[Your answers and explanations here.]

Your task is to complete the code that filters out extraneous noise and leaves only the birdsong. Choose a cutoff frequency `cutoff_freq` to filter the spectrogram.

To do this, set to zero all entries of `D_mixed` with frequencies below `cutoff_freq`. Note that this corresponds to the first index of `D_mixed` (not the second), even though it is plotted along the y-axis.

In [ ]:
#your code here
cutoff_freq = # your value here

# create a copy of D to modify
D_filtered = D_mixed.copy()

D_filtered = ...

In [106]:
# solved
fft_freqs = librosa.fft_frequencies(sr=sr_mixed)
cutoff_freq = 2048
# create a copy of the STFT to modify
D_filtered = D_mixed.copy()
# Find indices of frequencies below the cutoff
low_freq_indices = np.where(fft_freqs < cutoff_freq)[0]
# Set magnitudes of low frequencies to zero
D_filtered[low_freq_indices, :] = 0
# Convert the filtered STFT to decibels for visualization
S_db_filtered = librosa.amplitude_to_db(np.abs(D_filtered), ref=np.max)
# Perform inverse STFT to get the filtered audio time series
y_filtered = librosa.istft(D_filtered)
# solved

Plot the resulting spectrogram and play the sound. (To reconstruct `y_filtered` for playback, use the inverse transform `librosa.istft`.)

How did the audio track change?

In [ ]:
# your code here
print(f"Original mixed audio shape: {y_mixed.shape}")
print(f"Filtered audio shape: {y_filtered.shape}")

# Plot the filtered spectrogram
plt.figure(figsize=(12, 6))
librosa.display.specshow(S_db_filtered, sr=sr_mixed, x_axis='time', y_axis='log',
                         cmap='magma', fmax=sr_mixed/2)
plt.colorbar(format='%+2.0f dB')
plt.title(f'Spectrogram of Mixed Audio (Frequencies below {cutoff_freq} Hz removed)')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

# Play the filtered audio
print("Playing filtered mixed audio:")
Audio(data=y_filtered, rate=sr_mixed)
# your code here

[Your conclusions here.]